# Raquette — Training Notebook

Run this in Google Colab (Runtime → Change runtime type → **T4 GPU**).

This notebook will:
1. Install all dependencies
2. Download TrackNet pre-trained weights
3. Download a public tennis dataset
4. Extract pose sequences from the footage
5. Train the shot classifier
6. Save weights you download and drop into `ml/models/weights/`

**Estimated time: ~35 minutes on free Colab T4**

## 1 · Install dependencies

In [ ]:
!pip install -q ultralytics mediapipe opencv-python-headless gdown
!pip install -q torch torchvision --index-url https://download.pytorch.org/whl/cu118
print('✓ Dependencies installed')

## 2 · Clone Raquette + download TrackNet weights

In [ ]:
import os

# Clone your repo (replace with your GitHub URL once pushed)
# !git clone https://github.com/YOUR_USERNAME/raquette.git
# %cd raquette

# For now, create the weights directory
os.makedirs('ml/models/weights', exist_ok=True)

# Download TrackNet V2 weights (official release — non-commercial use)
# Source: https://github.com/TrackNetTeam/TrackNet
!gdown --id '1bsg3HBdSawyKCjpdhA6FVHO3byL05Jf8' -O ml/models/weights/tracknet.pt

# Verify
import os
size = os.path.getsize('ml/models/weights/tracknet.pt') / 1e6
print(f'✓ TrackNet weights: {size:.1f} MB')

## 3 · Download tennis dataset

We use the publicly available **TrackNet dataset** which contains annotated tennis match footage with ball positions. We'll extract pose sequences from it for shot classifier training.

Alternatively, record your own footage — 30+ minutes of match play is enough.

In [ ]:
import os

os.makedirs('data/videos', exist_ok=True)
os.makedirs('data/annotations', exist_ok=True)

# TrackNet dataset — Game1 through Game10
# Full dataset: https://nol.cs.nctu.edu.tw:234/open-source/TrackNet
# For this notebook we download a single game as a demo
print('Downloading TrackNet Game1...')
!wget -q 'https://nol.cs.nctu.edu.tw:234/open-source/TrackNet/data/Game1.zip' -O data/Game1.zip
!cd data && unzip -q Game1.zip && mv Game1 videos/
print('✓ Dataset ready')

import glob
videos = glob.glob('data/videos/**/*.mp4', recursive=True)
csvs   = glob.glob('data/videos/**/*.csv',  recursive=True)
print(f'  {len(videos)} video clips, {len(csvs)} annotation files')

## 4 · Extract pose sequences + auto-label shots

For each annotated ball position, we:
- Detect players with YOLOv8
- Extract MediaPipe pose at the hitting player
- Label shot type using a heuristic (wrist velocity + side of body) — **good enough for bootstrapping, then you refine**

In [ ]:
import cv2
import numpy as np
import pandas as pd
import mediapipe as mp
from ultralytics import YOLO
from pathlib import Path
import pickle

SHOT_LABELS = ['Forehand', 'Backhand', 'Serve', 'Volley', 'Smash', 'Slice']

yolo = YOLO('yolov8m.pt')
pose_est = mp.solutions.pose.Pose(static_image_mode=False, model_complexity=1)

def extract_joint_angles(landmarks):
    if not landmarks:
        return np.zeros(12)
    lm = np.array([(l.x, l.y, l.z) for l in landmarks])
    def angle(a, b, c):
        v1, v2 = lm[a]-lm[b], lm[c]-lm[b]
        cos = np.dot(v1,v2)/(np.linalg.norm(v1)*np.linalg.norm(v2)+1e-8)
        return np.degrees(np.clip(cos,-1,1))
    return np.array([
        angle(11,13,15), angle(12,14,16),
        angle(13,11,23), angle(14,12,24),
        angle(11,23,25), angle(12,24,26),
        lm[15,0]-lm[16,0], lm[15,1]-lm[16,1],
        lm[11,0]-lm[12,0], lm[23,0]-lm[24,0],
        lm[11,1]-lm[23,1], lm[12,1]-lm[24,1],
    ], dtype=np.float32)

def heuristic_shot_label(pose_window, ball_trajectory, side):
    """Rough heuristic — good enough for initial training data."""
    if not pose_window:
        return None
    last = pose_window[-1]
    lm = np.array([(l.x, l.y, l.z) for l in last])
    # High wrist above shoulder = serve or smash
    right_wrist_y = lm[16,1]
    right_shoulder_y = lm[12,1]
    if right_wrist_y < right_shoulder_y - 0.1:
        return 'Serve' if len(ball_trajectory) < 3 else 'Smash'
    # Wrist on dominant side = forehand, other side = backhand
    if side == 'right':
        return 'Forehand' if lm[16,0] > lm[11,0] else 'Backhand'
    else:
        return 'Backhand' if lm[15,0] > lm[11,0] else 'Forehand'

sequences = []  # list of (feature_array [16,12], label_int)

for csv_path in sorted(Path('data').glob('**/*.csv'))[:5]:  # first 5 clips
    video_path = csv_path.with_suffix('.mp4')
    if not video_path.exists():
        continue

    df = pd.read_csv(csv_path)
    cap = cv2.VideoCapture(str(video_path))
    pose_window, ball_traj = [], []
    prev_y = None

    for _, row in df.iterrows():
        cap.set(cv2.CAP_PROP_POS_FRAMES, int(row.get('Frame', 0)))
        ret, frame = cap.read()
        if not ret:
            continue

        bx, by = float(row.get('X', 0)), float(row.get('Y', 0))
        ball_visible = row.get('Visibility', 0) == 1

        if ball_visible:
            ball_traj.append((bx, by))

        # Contact detection: ball Y direction reversal
        is_contact = False
        if ball_visible and prev_y is not None:
            if (by - prev_y) * (prev_y - (ball_traj[-3][1] if len(ball_traj)>3 else by)) < 0:
                is_contact = True
        if ball_visible:
            prev_y = by

        # Pose extraction
        rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        result = pose_est.process(rgb)
        if result.pose_landmarks:
            pose_window.append(result.pose_landmarks.landmark)

        if is_contact and len(pose_window) >= 8:
            window = pose_window[-16:]
            features = np.array([extract_joint_angles(p) for p in window])
            # Pad to 16
            while len(features) < 16:
                features = np.vstack([features, features[-1:]])
            features = features[:16]

            # Determine dominant side from hip position relative to ball
            lm = np.array([(l.x, l.y, l.z) for l in pose_window[-1]])
            side = 'right' if lm[24,0] > bx/frame.shape[1] else 'left'

            label_str = heuristic_shot_label(pose_window[-4:], ball_traj, side)
            if label_str and label_str in SHOT_LABELS:
                sequences.append((features, SHOT_LABELS.index(label_str)))
            pose_window = pose_window[-4:]

    cap.release()
    print(f'  {csv_path.name}: {len(sequences)} sequences so far')

print(f'\n✓ Total sequences: {len(sequences)}')
label_counts = {SHOT_LABELS[i]: sum(1 for _,l in sequences if l==i) for i in range(6)}
print('  Distribution:', label_counts)

with open('data/sequences.pkl', 'wb') as f:
    pickle.dump(sequences, f)
print('✓ Saved to data/sequences.pkl')

## 5 · Train the shot classifier

In [ ]:
import pickle
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, random_split

with open('data/sequences.pkl', 'rb') as f:
    sequences = pickle.load(f)

print(f'Loaded {len(sequences)} sequences')

class ShotDataset(Dataset):
    def __init__(self, seqs):
        self.X = torch.tensor(np.stack([s[0] for s in seqs]), dtype=torch.float32)
        self.y = torch.tensor([s[1] for s in seqs], dtype=torch.long)
    def __len__(self): return len(self.X)
    def __getitem__(self, i): return self.X[i], self.y[i]

dataset = ShotDataset(sequences)
n_val = max(1, int(len(dataset) * 0.15))
train_ds, val_ds = random_split(dataset, [len(dataset)-n_val, n_val])

train_dl = DataLoader(train_ds, batch_size=32, shuffle=True)
val_dl   = DataLoader(val_ds,   batch_size=64)

class ShotClassifierModel(nn.Module):
    def __init__(self, input_size=12, num_classes=6):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv1d(input_size, 64, 3, padding=1), nn.ReLU(),
            nn.Conv1d(64, 128, 3, padding=1), nn.ReLU(),
            nn.AdaptiveAvgPool1d(4),
        )
        self.head = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128*4, 128), nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, num_classes),
        )
    def forward(self, x):
        return self.head(self.conv(x.permute(0,2,1)))

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Training on {device}')

model = ShotClassifierModel().to(device)
opt   = torch.optim.Adam(model.parameters(), lr=3e-4)
sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=30)
loss_fn = nn.CrossEntropyLoss()

best_val_acc = 0

for epoch in range(40):
    # Train
    model.train()
    train_loss = 0
    for X, y in train_dl:
        X, y = X.to(device), y.to(device)
        opt.zero_grad()
        loss = loss_fn(model(X), y)
        loss.backward()
        opt.step()
        train_loss += loss.item()

    # Validate
    model.eval()
    correct = total = 0
    with torch.no_grad():
        for X, y in val_dl:
            X, y = X.to(device), y.to(device)
            preds = model(X).argmax(1)
            correct += (preds == y).sum().item()
            total   += len(y)
    val_acc = correct / max(total, 1)
    sched.step()

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), 'ml/models/weights/shot_classifier.pt')

    if (epoch+1) % 5 == 0:
        print(f'Epoch {epoch+1:3d} | loss {train_loss/len(train_dl):.4f} | val acc {val_acc:.2%}')

print(f'\n✓ Best val accuracy: {best_val_acc:.2%}')
print('✓ Weights saved to ml/models/weights/shot_classifier.pt')

## 6 · Fine-tune YOLOv8 for tennis players (optional)

The base `yolov8m.pt` already detects people well. Run this cell only if you want a tennis-specific detector (useful if your footage has spectators in frame).

In [ ]:
# OPTIONAL — skip if base YOLO is good enough
from ultralytics import YOLO

# You need a labelled dataset in YOLO format.
# Quick approach: use Roboflow to label 200-300 frames, export as YOLOv8.
# Then:
#   model = YOLO('yolov8m.pt')
#   model.train(data='your_dataset.yaml', epochs=30, imgsz=640, device=0)
#   model.export()  # saves best.pt
#   !cp runs/detect/train/weights/best.pt ml/models/weights/player_detector.pt

print('Skipping fine-tune — base yolov8m.pt works well for most footage.')
print('To use base weights: the pipeline auto-downloads yolov8m.pt on first run.')

## 7 · Download weights

Download `shot_classifier.pt` and optionally `tracknet.pt`, then drop them into:
```
ml/models/weights/
  shot_classifier.pt
  tracknet.pt
```

In [ ]:
from google.colab import files
files.download('ml/models/weights/shot_classifier.pt')
files.download('ml/models/weights/tracknet.pt')
print('✓ Check your browser downloads folder')

## Done!

Place the downloaded `.pt` files in `ml/models/weights/` on your machine, then start the backend:

```bash
cd /path/to/Raquette
.venv/bin/uvicorn backend.app.main:app --reload --port 8000
```

The pipeline will automatically use the real models instead of the simulation.